setup:
- please make sure that mochi_class is the first classy instance avialble on PATH
- this tutorial should be used with the 292-mochi-class branch of cloelib

In [1]:
# General imports
import numpy as np
import matplotlib.pyplot as plt

# cloelite imports: we load the following interfaces
from cloelib.cosmology.mochi_class_cosmology import mochiCLASSBackground, mochiCLASSLinearPerturbations, mochiCLASSNonLinearPerturbations # requires cloelib branch 292-mochi-class
from cloelib.cosmology.camb_cosmology import CAMBBackground, CAMBLinearPerturbations, CAMBNonLinearPerturbations
from cloelib.cosmology.HMcode2020Emu_cosmology import HMemuLinearPerturbations, HMemuNonLinearPerturbations
from cloelib.cosmology.jax_cosmology import JAXBackground, JAXLinearPerturbations, JAXNonLinearPerturbations
from cloelib.cosmology.cosmology import Background

# Validation benchmark in astropy
import astropy.units as u

# for solving M2 ODE (Lombriser parameterisation)
from scipy.integrate import solve_ivp
from scipy.interpolate import UnivariateSpline

# mochi_class only

load example files for stable basis parameterisation (galileon) from mochi_class github

In [2]:
# import stable parameters from file
# mochi - cubic galileon
# use the lna array from this example for our stable basis too
lna_smg_gal, Delta_Mpl_gal, Dkin_gal, cs2_gal = np.loadtxt("cubic_galileon_stable_params_mathematica.dat", unpack=True)
lna_de_gal, rho_de_gal = np.loadtxt("rho_de_stable_cubic_galileon_mathematica.dat", unpack=True, dtype=float)
alpha_B0_gal =  1.380051990990777

## testing the mochi_class implementation (just wowaCDM, no MG yet)

In [ ]:
# Cosmology parameters
H0 =  67.7
h = H0/100.
sigma8 = 0.8277
omch2 = 0.12
Omega_cdm0 = omch2/h**2
ombh2 = 0.022
Omega_b0 = ombh2/h**2
Omega_k0 = 0.
w = -0.8
wa = 0.
ns = 0.96
mnu = 0.
As=2e-9

# z-array
z = np.linspace(0, 3, 100)

stable_MG_dict = {
    "w0": w,
    "wa": wa, 
    "lna_smg": lna_smg_gal, # same lna spacing as in ,mochi_class example files
}

# mochi_class background (w0wa cosmology, no MG)
class_instance = mochiCLASSBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                                Omega_k0=Omega_k0,
                                As=As, ns=ns, mnu=0., #w0=-1.0, wa=0.0, 
                                N_mnu=0,
                                mg_stable_basis_on=False,
                                mg_background_model='wowa',
                                stable_MG_dict = stable_MG_dict,
                                )
H_z_class = class_instance.hubble_parameter(z)
chi_z_class = class_instance.comoving_distance(z)
dA_z_class = class_instance.angular_diameter_distance(z)
tr_z_class = class_instance.transverse_comoving_distance(z)

# comparison with JAX
jax_instance = JAXBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                                Omega_k0=Omega_k0,
                                As=As, ns=ns, mnu=0., w0=w, wa=wa, 
                                N_mnu=0,
                                gamma_MG=0.0
                                )
H_z_jax = jax_instance.hubble_parameter(z)
chi_z_jax = jax_instance.comoving_distance(z)
dA_z_jax = jax_instance.angular_diameter_distance(z)
tr_z_jax = jax_instance.transverse_comoving_distance(z)

fig, axs = plt.subplots(figsize = (16,14), nrows=3, ncols=1)
axs[0].plot(z, H_z_jax, label = 'jax', ls='-')
axs[0].plot(z, H_z_class, label = 'CLASS', ls='--')
axs[0].set_ylabel('H(z)', fontsize = 16)
axs[0].set_xlabel('Redshift z', fontsize = 16)
axs[0].legend()
axs[1].plot(z, chi_z_jax, label = 'jax chi_z', ls='-')
axs[1].plot(z, chi_z_class, label = 'CLASS chi_z', ls='--')
axs[1].set_ylabel('Comoving distance (Mpc)', fontsize = 16)
axs[1].set_xlabel('Redshift z', fontsize = 16)
axs[1].legend()
axs[2].plot(z, dA_z_jax*(1+z), label = 'jax d_A', ls='-')
axs[2].plot(z, dA_z_class*(1+z), label = 'CLASS d_A', ls='--')
axs[2].set_ylabel('Distance (Mpc)', fontsize = 16)
plt.gca().set_prop_cycle(None)
axs[2].plot(z, tr_z_jax, label = 'jax tr_chi_z', ls='-')
axs[2].plot(z, tr_z_class, label = 'CLASS tr_chi_z', ls='--')
axs[2].set_xlabel('Redshift z', fontsize = 16)
axs[2].legend()


In [ ]:
# PERTURBATIONS
ks = np.logspace(np.log10(1e-4), np.log10(5), 100)

# mochi CLASS PERTURBATIONS
class_linear = mochiCLASSLinearPerturbations(background=class_instance, redshifts=z)
class_nonlinear = mochiCLASSNonLinearPerturbations(background=class_instance, 
                                              redshifts=z, 
                                              nonlinear_model='hmcode16')
linear_pk_class = class_linear.matter_power_spectrum(z, ks)
nonlinear_pk_class = class_nonlinear.matter_power_spectrum(z, ks)
plt.loglog(ks, linear_pk_class[0, :], label= 'mochi linear')
plt.loglog(ks, nonlinear_pk_class[0, :], label= 'mochi nonlinear')

# jax perturbations
jax_linear = JAXLinearPerturbations(background=jax_instance)
jax_nonlinear = JAXNonLinearPerturbations(background=jax_instance) 
linear_pk_jax = jax_linear.matter_power_spectrum(z, ks)
nonlinear_pk_jax = jax_nonlinear.matter_power_spectrum(z, ks)
plt.loglog(ks, linear_pk_jax[0, :], 'k--',label= 'jax linear')
plt.loglog(ks, nonlinear_pk_jax[0, :], 'k--', label= 'jax nonlinear')

plt.xlabel('k (1/Mpc)', fontsize = 16)
plt.ylabel('P(k)', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("Perturbations", fontsize = 16)


## stable basis prameterisation (Lombriser+ 2019) in mochiclass

### define functions parameterisation

In [5]:
def Solve_M2_ODE(lna_array, 
                 H, rho_phi,
                 Dkin, cs2, b, # stable basis 
                 M2_init = 1., M2D_init = 0., # initial conditions for ODE
                 w0=-1, wa=0,):
    '''
    solve ODE for M2 (= M^2) numerically
    M2'' + r (M2')^2/M2 + p M2' + q M2 = g
    with r,p,q,g as functions of x=lna
    solve forward, initioal conditions at lna=-5
    '''
    # H_dot = dH/dlna
    H_spline = UnivariateSpline(lna_array, H, s=0)
    H_dot = H_spline.derivative()(lna_array)

    # Calculate w(a) using the w0wa model
    w_array = w0 + wa * (1 - np.exp(lna_array))
    
    # Calculate coefficients
    # derived from mochi cs2 eqn
    r = 1/b -2
    p_array = 1 - b + H_dot/H
    q_array = b*H_dot/H + b/2 * cs2*Dkin
    g_array = 3 *(b/(2*H**2)*(rho_phi + w_array*rho_phi) + 1/3 * b * H_dot/H)
    

    # Lombriser 2018 paper -> tested, equivalent to eqn above
    '''
    r = (1 - 2 * b) / b
    p_array = -1/2 * (1 + 2 * b + 3 * Omega_DE * w_array)
    q_array = b/2 * (cs2 * Dkin - 3 - 3 * Omega_DE * w_array)
    g_array = b/2 * (3 * Omega_DE - 3)
    '''
    
    # Interpolation functions for p, q, g    
    def p(lna):
        return np.interp(lna, lna_array, p_array)

    def q(lna):
        return np.interp(lna, lna_array, q_array)

    def g(lna):
        return np.interp(lna, lna_array, g_array)

    # Define the ODE system: M2' = D, D' = [Original ODE in terms of M2 and D]
    def system(lna, Y):
        M2, D = Y # Y is the vector [M2, M2']
        dM2_dlna = D
        dD_dlna = g(lna) - q(lna) * M2 - p(lna)*D - r * (D**2) / M2 
        return [dM2_dlna, dD_dlna]

    # Initial conditions
    # Set your initial condition for M2 and M2'
    initial_conditions = [M2_init, M2D_init]
    #initial_conditions = [1, 1]

    # Solve the ODE
    try:
        solution = solve_ivp(system, t_span = (lna_array[0], lna_array[-1]), y0 = initial_conditions, t_eval=lna_array, method='RK45',rtol = 1e-7 , atol = 1e-10) # higher accuracy crucial, cause small deviations will be magnified by 1/Dkin later on
        print('size lna_array', len(lna_array))
        print('size M2 array', len(solution.y[0]))
        
        if solution.status == -1:
            raise ValueError("Integration failed.")

    except ValueError as e:
            raise 

    return solution



def stable_basis_from_M2_ODE( H0, Omega_b0, Omega_cdm0, Omega_k0, As, ns, # cosmological parameters
                                lna_mochi, # lna range for basis functions
                                w0 = -1, wa = 0, # background
                                b = 2, # M2 parameterisation, b is either one or two
                                s = 0, # cs2 parameterisation
                                a0 = 0, a1 = 0, u1 = 1, u2 = 1,# alpha parameterisation
                                M2_init=1, # initial condition for M2 at lna=-5
                                M2D_init=0, # initial condition for M2' at lna=-5
                            ):
    ''' 
    Lombriser et al 2018 parameterisation of stable basis functions (eqn 4.4-4.7)
    alpha is equiv to Dkin
    solve M2 ODE to incorporate alpha_B ~ alpha_M proportionality
    here alpha_M = d ln M2/ d ln a = 1/M2 d M2/ d ln a = M2' / M2 
    lna_mochi: lna range for basis functions - here using internal mochi spacing

    output:
        dictionary with stable basis functions, to be fed into mochi_class cloelib protocol
    '''

    ###############################################
    # get Omega_DE from background - ONLY FOR wowa
    ###############################################
    stable_MG_dict = {
    "w0": w0,
    "wa": wa, 
    }

    # background for w0wa (MG off)
    background_instance = mochiCLASSBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                                    Omega_k0=Omega_k0,
                                    As=As, ns=ns, mnu=0.,
                                    N_mnu=0,
                                    mg_stable_basis_on=False,
                                    mg_background_model='wowa',
                                    stable_MG_dict = stable_MG_dict,
                                    )
    
    # compute Omega_DE from background for parameterisation, and get background evolution for M2 ODE
    background_wowa = background_instance.get_background() 
    lna_wowa = np.log(1/(1 + background_wowa['z'])) # lna-spacing for all time functions, output by mochi
    
    # rho_fld if just wowa (no mg)
    H_wowa_lna = background_wowa['H [1/Mpc]'] # Hubble in 1/Mpc
    rhoDE_wowa_lna = background_wowa['(.)rho_fld'] # only valid when flg used in mochi background instance (i.e. no-MG)
    rhoCrit_wowa_lna = background_wowa['(.)rho_crit']
    Omega_DE_lna = rhoDE_wowa_lna/rhoCrit_wowa_lna
    
    # interpolate for late times (MG scale factors)
    Omega_DE = UnivariateSpline(lna_wowa, Omega_DE_lna , s=0)(lna_mochi)
    H_wowa = UnivariateSpline(lna_wowa, H_wowa_lna, s=0)(lna_mochi)
    rhoDE_wowa = UnivariateSpline(lna_wowa, rhoDE_wowa_lna , s=0)(lna_mochi)
    
    ###############################################
    ####### parametertisation stable basis ########
    ###############################################
    cs2_param = 1 + s * Omega_DE/Omega_DE[-1]
    alpha_param= (a0*(1+s) + a1 * (1- np.exp(lna_mochi)**u1)) * (Omega_DE/Omega_DE[-1])**u2 / cs2_param
    
    ############# compute M2 from ODE #############
    if b==0:
        M2_param = np.ones_like(cs2_param)
        alpha_M = np.zeros_like(cs2_param)
        alpha_B_propto = np.zeros_like(cs2_param)
    else:
        M2_solution = Solve_M2_ODE(lna_mochi, H_wowa, rhoDE_wowa,
                                alpha_param, cs2_param, b, 
                                w0=w0, wa=wa,
                                M2_init= M2_init, M2D_init=M2D_init,
                                )
        M2_param = M2_solution.y[0] # y[0] is M2
        alpha_M = M2_solution.y[1]/M2_param # y[1] is M2' (dM2/dlna)
        alpha_B_propto = - 2/b * alpha_M # mochi alpha_B (= - 2* alpha_B_Lombriser)

    alpha_B0 = alpha_B_propto[-1] # alpha_B0 is the value today (end of lna array)
    print('alpha_B0 from ODE solution:', alpha_B0)
    Delta_Mpl_param = M2_param - 1 # Delta_Mpl is M2 - 1

    ###############################################
    ####### collect stable basis in dict ########
    ###############################################
    stable_basis_dictionary = {
        # background 
        "lna_smg": lna_mochi,
        # stable basis
        "D_kin": alpha_param,
        "cs2": cs2_param,
        "Delta_M2": Delta_Mpl_param,
        "alpha_B0": alpha_B0, # output of mochi
        "alpha_M": alpha_M,
        "w0": w0,
        "wa": wa, 
    }

    return stable_basis_dictionary

### compute mochi_class output for this stable basis parameterisation (wowa, with MG)

In [ ]:
# stable basis parameters
b = 1 # M2 parameterisation, b is either one or two
s = -0.3 # cs2 parameterisation
# alpha parameterisation
a0 = 0.2
a1 = 0.2
u1 = 1
u2 = 1

# lna array (spacing 'time' evolution)
lna_mochi = lna_smg_gal

stable_basis_dict_updated = stable_basis_from_M2_ODE( H0, Omega_b0, Omega_cdm0, Omega_k0, As, ns, # cosmological parameters
                                lna_mochi, # lna range for basis functions
                                w0 = w, wa = wa, # background
                                b = b, # M2 parameterisation, b is either one or two
                                s = s, # cs2 parameterisation
                                a0 = a0, a1 = a1, u1 = u1, u2 = u2,# alpha parameterisation
                                M2_init=1, # initial condition for M2 at lna=-5
                                M2D_init=0, # initial condition for M2' at lna=-5
                            )

# z-array
z = np.linspace(0, 3, 100)

# background
mochiclass_instance = mochiCLASSBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                                Omega_k0=Omega_k0,
                                As=As, ns=ns, mnu=0., #w0=-1.0, wa=0.0, 
                                N_mnu=0,
                                mg_stable_basis_on=True,
                                mg_background_model='wowa',
                                stable_MG_dict = stable_basis_dict_updated,
                                )

mochiclass_lcdm_instance = mochiCLASSBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                                Omega_k0=Omega_k0,
                                As=As, ns=ns, mnu=0., #w0=-1.0, wa=0.0, 
                                N_mnu=0,
                                mg_stable_basis_on=False,
                                mg_background_model='lcdm',
                                stable_MG_dict = stable_MG_dict,
                                )
H_z_mochiclass = mochiclass_instance.hubble_parameter(z,'km/s/Mpc')
H_z_mochiclass_lcdm = mochiclass_lcdm_instance.hubble_parameter(z,'km/s/Mpc')
plt.plot(z, H_z_mochiclass, label = 'MG')
plt.plot(z, H_z_mochiclass_lcdm, label = 'LCDM')
plt.xlabel('Redshift z', fontsize = 16)
plt.ylabel('H(z) km/s/Mpc', fontsize = 16)
plt.legend(fontsize = 16)
plt.show()
# perturbations
ks = np.logspace(np.log10(1e-4), np.log10(5), 100)

# CLASS PERTURBATIONS
# k-array
mochiclass_linear = mochiCLASSLinearPerturbations(background=mochiclass_instance, redshifts=z, ks=ks)
mochiclass_lcdm_linear = mochiCLASSLinearPerturbations(background=mochiclass_lcdm_instance, redshifts=z, ks=ks)

mochiclass_nonlinear = mochiCLASSNonLinearPerturbations(background=mochiclass_instance, 
                                              redshifts=z, 
                                              ks=ks,
                                              nonlinear_model='halofit')
mochiclass_lcdm_nonlinear = mochiCLASSNonLinearPerturbations(background=mochiclass_lcdm_instance, 
                                              redshifts=z, 
                                              ks=ks,
                                              nonlinear_model='halofit')
linear_pk_mochiclass = mochiclass_linear.matter_power_spectrum(z, ks)
linear_pk_mochiclass_lcdm = mochiclass_lcdm_linear.matter_power_spectrum(z, ks)
nonlinear_pk_mochiclass = mochiclass_nonlinear.matter_power_spectrum(z, ks)
nonlinear_pk_mochiclass_lcdm = mochiclass_lcdm_nonlinear.matter_power_spectrum(z, ks)

# plt.loglog(ks, linear_pk_mochiclass[0, :], label= 'linear')
# plt.loglog(ks, linear_pk_mochiclass_lcdm[0, :], label= 'linear, lcdm')
plt.loglog(ks, nonlinear_pk_mochiclass[0, :], label= 'nonlinear MG')
plt.loglog(ks, nonlinear_pk_mochiclass_lcdm[0, :], label= 'nonlinear LCDM')
plt.xlabel('k (1/Mpc)', fontsize = 16)
plt.ylabel('P(k)', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("Nonlinear Pk", fontsize = 16)

# compare cosmologies (no-MG)

## Background quantities

In [ ]:
# Cosmology parameters
H0 =  67.7
h = H0/100.
sigma8 = 0.8277
omch2 = 0.12
Omega_cdm0 = omch2/h**2
ombh2 = 0.022
Omega_b0 = ombh2/h**2
Omega_k0 = 0.
w = -1.
wa = 0.
ns = 0.96
mnu = 0.
As=2e-9

# Astropy
Tcmb = 2.7255
Neff = 0.0#3.046
YHe = 0.2454

# z-array
z = np.linspace(0, 3, 100)

astropy_instance = FlatLambdaCDM(H0=H0,Om0=(Omega_cdm0 + Omega_b0),Ob0=Omega_b0, Tcmb0=Tcmb,Neff=Neff, m_nu=[0,0,0.0]*u.eV)
H_z_astropy = astropy_instance.H(z).to(u.km/u.s/u.Mpc).value
chi_z_astropy = astropy_instance.comoving_distance(z)
dA_z_astropy = astropy_instance.angular_diameter_distance(z)
tr_z_astropy = astropy_instance.comoving_transverse_distance(z)

jax_instance = jaxBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                               Omega_k0=Omega_k0,
                               As=As, ns=ns, mnu=0., w0=-1.0, wa=0.0, 
                               gamma_MG=0.0, N_mnu=0)
H_z_jax = jax_instance.hubble_parameter(z, units='km/s/Mpc')
chi_z_jax = jax_instance.comoving_distance(z)
dA_z_jax = jax_instance.angular_diameter_distance(z)
tr_z_jax = jax_instance.transverse_comoving_distance(z)

class_instance = CLASSBackground(H0=H0, Omega_b0=Omega_b0, Omega_cdm0=Omega_cdm0, 
                               Omega_k0=Omega_k0,
                               As=As, ns=ns, mnu=0., w0=-1.0, wa=0.0, 
                               gamma_MG=0.0, N_mnu=0)
H_z_class = class_instance.hubble_parameter(z)
chi_z_class = class_instance.comoving_distance(z)
dA_z_class = class_instance.angular_diameter_distance(z)
tr_z_class = class_instance.transverse_comoving_distance(z)

jax_instance = JAXBackground(H0=H0,Omega_b0=Omega_b0,Omega_cdm0=Omega_cdm0, 
                             Omega_k0=Omega_k0,
                             As=As, ns=ns, mnu=mnu, w0=w, wa=wa, gamma_MG=0., N_mnu=0)
H_z_jax = jax_instance.hubble_parameter(z)
chi_z_jax = jax_instance.comoving_distance(z)
dA_z_jax = jax_instance.angular_diameter_distance(z)
tr_z_jax = jax_instance.transverse_comoving_distance(z)

fig, axs = plt.subplots(figsize = (16,14), nrows=3, ncols=1)
axs[0].plot(z, H_z_jax, label = 'jax', ls='--')
axs[0].plot(z, H_z_class, label = 'CLASS', ls='-.')
axs[0].plot(z, H_z_astropy, label = 'Astropy', ls=':')
axs[0].plot(z, H_z_jax, label = 'JAX', ls='-')
axs[0].set_ylabel('H(z)', fontsize = 16)
axs[0].set_xlabel('Redshift z', fontsize = 16)
axs[0].legend()
axs[1].plot(z, chi_z_jax, label = 'jax chi_z', ls='--')
axs[1].plot(z, chi_z_class, label = 'CLASS chi_z', ls='-.')
axs[1].plot(z, chi_z_astropy, label = 'Astropy chi_z', ls=':')
axs[1].plot(z, chi_z_jax, label = 'JAX chi_z', ls='-')
axs[1].set_ylabel('Comoving distance (Mpc)', fontsize = 16)
axs[1].set_xlabel('Redshift z', fontsize = 16)
axs[1].legend()
axs[2].plot(z, dA_z_jax*(1+z), label = 'jax d_A', ls='-')
axs[2].plot(z, dA_z_class*(1+z), label = 'CLASS d_A', ls='-')
axs[2].plot(z, dA_z_astropy*(1+z), label = 'Astropy d_A', ls='-')
axs[2].plot(z, dA_z_jax*(1+z), label = 'JAX d_A', ls='-')
axs[2].set_ylabel('Distance (Mpc)', fontsize = 16)
plt.gca().set_prop_cycle(None)
axs[2].plot(z, tr_z_jax, label = 'jax tr_chi_z', ls='--')
axs[2].plot(z, tr_z_class, label = 'CLASS tr_chi_z', ls='--')
axs[2].plot(z, tr_z_astropy, label = 'Astropy tr_chi_z', ls='--')
axs[2].plot(z, tr_z_jax, label = 'JAX tr_chi_z', ls='--')
axs[2].set_xlabel('Redshift z', fontsize = 16)
axs[2].legend()

#Differences are due to neutrinos

## perturbations

In [3]:
ks = np.logspace(np.log10(1e-4), np.log10(5), 100)

In [ ]:
# JAX PERTURBATIONS
# k-array
jax_linear= JAXLinearPerturbations(background=jax_instance)
jax_nonlinear = JAXNonLinearPerturbations(background=jax_instance)
linear_pk_jax = jax_linear.matter_power_spectrum(z, ks)
nonlinear_pk_jax = jax_nonlinear.matter_power_spectrum(z, ks)
plt.loglog(ks, linear_pk_jax[0, :], label= 'linear')
plt.loglog(ks, nonlinear_pk_jax[0, :], label= 'nonlinear')
plt.xlabel('k (1/Mpc)', fontsize = 16)
plt.ylabel('P(k)', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("JAX Perturbations", fontsize = 16)

In [ ]:
# jax PERTURBATIONS
jax_linear = jaxLinearPerturbations(background=jax_instance, redshifts=z)
jax_nonlinear = jaxNonLinearPerturbations(background=jax_instance, redshifts=z, 
                                            nonlinear_model='mead2016')

linear_pk_jax = jax_linear.matter_power_spectrum(z, ks)
nonlinear_pk_jax = jax_nonlinear.matter_power_spectrum(z, ks)
plt.loglog(ks, linear_pk_jax[0, :], label= 'linear')
plt.loglog(ks, nonlinear_pk_jax[0, :], label= 'nonlinear')
plt.xlabel('k (1/Mpc)', fontsize = 16)
plt.ylabel('P(k)', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("jax Power Spectrum", fontsize = 16)

In [ ]:
# CLASS PERTURBATIONS
# k-array
class_linear = CLASSLinearPerturbations(background=class_instance, redshifts=z)
class_nonlinear = CLASSNonLinearPerturbations(background=class_instance, 
                                              redshifts=z, 
                                              nonlinear_model='hmcode16')
linear_pk_class = class_linear.matter_power_spectrum(z, ks)
nonlinear_pk_class = class_nonlinear.matter_power_spectrum(z, ks)
plt.loglog(ks, linear_pk_class[0, :], label= 'linear')
plt.loglog(ks, nonlinear_pk_class[0, :], label= 'nonlinear')
plt.xlabel('k (1/Mpc)', fontsize = 16)
plt.ylabel('P(k)', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("CLASS Perturbations", fontsize = 16)

In [ ]:
#calculate results for these parameters
plt.plot(z, jax_linear.growth_rate(), label = 'f(z) jax', ls='-.')
plt.plot(z, jax_linear.growth_rate(z), label = 'f(z) JAX', ls=':')
plt.plot(z, class_linear.growth_rate(), label = 'f(z) CLASS', ls='--')
plt.plot(z, jax_linear.growth_factor(z,ks)[:, 0], label = 'D(z) jax', ls='-.')
plt.plot(z, jax_linear.growth_factor(z), label = 'D(z) JAX', ls=':')
plt.plot(z, class_linear.growth_factor(z,ks)[:, 0], label = 'D(z) CLASS', ls='--')
plt.xlabel('z', fontsize = 16)
plt.legend(fontsize = 16)
plt.title("jax vs. CLASS vs. JAX Linear Perturbations")